# 🐼 Pandas en Google Colab · Parte 2 — Programación en Python · Solemne 1

Continuación de la Parte 1. Basado en los ejemplos `08` a `11` del repositorio **CursoPython**.

**Cómo usarlo:** ejecuta las celdas en orden con **Shift + Enter**.

| Unidad | Tema | Ejemplos del repo |
|---|---|---|
| 4 | Combinación de tablas: joins, merges y concatenaciones | `08_merge_tablas.py`, `09_granularidad_concat.py` |
| 5 | Limpieza de datos reales y automatización | `10_limpieza_datos.py`, `11_funcion_limpieza.py` |
| — | Ejercicios con comprobación automática + soluciones | `ejercicios_parte2.py` |

## 0. Cargar las cinco tablas del curso
Todas se leen directamente desde GitHub.

In [ ]:
import unicodedata
import pandas as pd

BASE = ("https://raw.githubusercontent.com/"
        "hpizarrog-industry40/CursoPython/main/Aprendiendo_Pandas/datos/")

alumnos    = pd.read_csv(BASE + "alumnos.csv")             # 1 fila por alumno
cursos     = pd.read_csv(BASE + "cursos.csv")              # 1 fila por curso
asistencia = pd.read_csv(BASE + "asistencia_mensual.csv")  # 1 fila por alumno y mes
periodo2   = pd.read_csv(BASE + "alumnos_2026_2.csv")      # segundo período
sucio      = pd.read_csv(BASE + "alumnos_sucio.csv")       # datos "reales" sin limpiar

for nombre, t in [("alumnos", alumnos), ("cursos", cursos), ("asistencia", asistencia),
                  ("periodo2", periodo2), ("sucio", sucio)]:
    print(f"{nombre:<11} {t.shape}")

---
## 4. Combinación de tablas
Tres conceptos guían toda la unidad:
- **Clave:** la columna que empareja filas (`curso`, `id`).
- **Granularidad:** qué representa una fila en cada tabla.
- **Trazabilidad:** poder decir de dónde viene cada fila del resultado.

### 4.1 merge: agregar columnas de una tabla auxiliar (`08_merge_tablas.py`)

In [ ]:
cursos

In [ ]:
df = alumnos.merge(cursos, on="curso", how="left")
print("Filas antes:", len(alumnos), "-> después:", len(df))
df[["nombre", "curso", "profesor", "sala", "creditos"]].head()

### 4.2 Tipos de join
`cursos` incluye **Biologia**, un curso sin alumnos. Observa cómo cambia el número de filas.

In [ ]:
for tipo in ["inner", "left", "right", "outer"]:
    print(f"how='{tipo}':", len(alumnos.merge(cursos, on="curso", how=tipo)), "filas")

### 4.3 Trazabilidad con `indicator=True` y control con `validate`

In [ ]:
df_outer = alumnos.merge(cursos, on="curso", how="outer", indicator=True)
df_outer["_merge"].value_counts()

In [ ]:
df_outer[df_outer["_merge"] == "right_only"][["curso", "profesor", "_merge"]]

In [ ]:
# validate="many_to_one": la clave debe ser única en la tabla de la derecha
df_ok = alumnos.merge(cursos, on="curso", how="left", validate="many_to_one")
df_ok.shape

### 4.4 Granularidad: el error más común al unir tablas (`09_granularidad_concat.py`)
`asistencia_mensual` tiene **3 filas por alumno**. ¿Qué pasa si la unimos tal cual?

In [ ]:
asistencia.head(6)

In [ ]:
malo = alumnos.merge(asistencia, on="id", how="left")
print("De", len(alumnos), "a", len(malo), "filas  ⚠️ cada alumno quedó repetido 3 veces")

In [ ]:
# validate detecta el problema antes de que contamine el análisis
try:
    alumnos.merge(asistencia, on="id", validate="one_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", str(e).splitlines()[0])

**Solución:** llevar la tabla auxiliar a la misma granularidad (1 fila por alumno) **antes** de unir.

In [ ]:
asist_alumno = (asistencia.groupby("id", as_index=False)["asistencia"].mean()
                .rename(columns={"asistencia": "asistencia_mensual_prom"}))
bueno = alumnos.merge(asist_alumno, on="id", how="left", validate="one_to_one")
print("Filas:", len(bueno))
bueno[["nombre", "asistencia", "asistencia_mensual_prom"]].head()

### 4.5 concat: apilar períodos
`merge` agrega **columnas**; `concat` agrega **filas**. Antes de apilar, comprobamos que las columnas coincidan
y agregamos una columna `periodo` para no perder la trazabilidad.

In [ ]:
print("Mismas columnas:", list(alumnos.columns) == list(periodo2.columns))

historico = pd.concat([alumnos.assign(periodo="2026-1"),
                       periodo2.assign(periodo="2026-2")],
                      ignore_index=True)
print("Filas:", len(alumnos), "+", len(periodo2), "=", len(historico))
historico["periodo"].value_counts()

In [ ]:
# Pregunta analítica: ¿cómo cambió la nota final promedio por curso?
comparacion = historico.pivot_table(index="curso", columns="periodo",
                                    values="nota_final", aggfunc="mean").round(2)
comparacion

In [ ]:
comparacion.plot(kind="bar", rot=0, title="Nota final promedio por curso y período");

### 📝 Registro de trazabilidad (completa esta celda)
- Tabla principal: `alumnos` (20 filas, clave `id`).
- Unión con `cursos`: how = …, filas resultado = …, filas `right_only` = …
- Asistencia mensual: se agregó por … antes de unir porque …
- Histórico: … filas, identificadas por la columna …

---
## 5. Limpieza de datos reales y automatización
### 5.1 Diagnóstico de `alumnos_sucio.csv` (`10_limpieza_datos.py`)

In [ ]:
sucio.head(8)

In [ ]:
print("Columnas:", list(sucio.columns))
print("Filas duplicadas:", sucio.duplicated().sum())
print("Formas distintas de escribir el curso:", sucio["Curso"].nunique())
sucio.dtypes

### 5.2 Paso a paso

In [ ]:
df = sucio.copy()

# 1) Nombres de columnas
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"\s*\(%\)", "", regex=True)
              .str.replace(" ", "_"))
df = df.rename(columns={"nota_parcial_1": "nota_parcial1", "nota_parcial_2": "nota_parcial2"})

# 2) Duplicados
df = df.drop_duplicates().reset_index(drop=True)

# 3) Texto
df["nombre"] = df["nombre"].str.strip().str.replace(r"\s+", " ", regex=True).str.title()
df["ciudad"] = df["ciudad"].str.strip().str.title()
list(df.columns), len(df)

In [ ]:
# 4) Categorías: minúsculas + sin tildes + mapeo de abreviaturas
def quitar_tildes(texto):
    if not isinstance(texto, str):
        return texto
    return "".join(c for c in unicodedata.normalize("NFKD", texto)
                   if not unicodedata.combining(c))

print("ANTES:", df["curso"].nunique(), "variantes")
df["curso"] = (df["curso"].str.strip().str.lower().apply(quitar_tildes)
               .replace({"mate": "matematicas", "fis.": "fisica"})
               .str.title())
print("DESPUÉS:", df["curso"].nunique(), "variantes")
df["curso"].value_counts()

In [ ]:
# 5) Números guardados como texto: "7,5", "N/A", "-", "95%"
for col in ["nota_parcial1", "nota_parcial2", "nota_final", "asistencia"]:
    texto = (df[col].astype(str).str.replace(",", ".", regex=False)
             .str.replace("%", "", regex=False).str.strip())
    df[col] = pd.to_numeric(texto, errors="coerce")   # lo que no es número -> NaN
df.dtypes

In [ ]:
# 6) Validación final
assert df["id"].is_unique, "Hay ids repetidos"
assert df["nota_final"].between(0, 10).all(), "Notas fuera de rango"
print("✅ Validaciones OK")
df.isnull().sum()

### 5.3 De pasos sueltos a un flujo reutilizable (`11_funcion_limpieza.py`)
Cada función hace **una** tarea y devuelve un DataFrame nuevo. `.pipe()` las encadena como una receta.

In [ ]:
def limpiar_nombres_columnas(df, renombrar=None):
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r"\s*\(%\)", "", regex=True)
                  .str.replace(" ", "_"))
    return df.rename(columns=renombrar) if renombrar else df

def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def estandarizar_categoria(df, columna, mapeo=None):
    df = df.copy()
    serie = df[columna].str.strip().str.lower().apply(quitar_tildes)
    if mapeo:
        serie = serie.replace(mapeo)
    df[columna] = serie.str.title()
    return df

def convertir_a_numero(df, columnas):
    df = df.copy()
    for col in columnas:
        texto = (df[col].astype(str).str.replace(",", ".", regex=False)
                 .str.replace("%", "", regex=False).str.strip())
        df[col] = pd.to_numeric(texto, errors="coerce")
    return df

MAPEO_CURSOS = {"mate": "matematicas", "fis.": "fisica"}
RENOMBRAR = {"nota_parcial_1": "nota_parcial1", "nota_parcial_2": "nota_parcial2"}
NUMERICAS = ["nota_parcial1", "nota_parcial2", "nota_final", "asistencia"]

def limpiar_datos(df):
    """Flujo completo. Devuelve (df_limpio, registro)."""
    registro = {"filas_entrada": len(df)}
    limpio = (df.pipe(limpiar_nombres_columnas, RENOMBRAR)
                .pipe(quitar_duplicados)
                .pipe(estandarizar_categoria, "curso", MAPEO_CURSOS)
                .pipe(estandarizar_categoria, "ciudad")
                .pipe(convertir_a_numero, NUMERICAS))
    limpio["nombre"] = limpio["nombre"].str.strip().str.replace(r"\s+", " ", regex=True).str.title()
    registro["duplicados_eliminados"] = registro["filas_entrada"] - len(limpio)
    registro["filas_salida"] = len(limpio)
    registro["faltantes"] = limpio.isnull().sum()[lambda s: s > 0].to_dict()
    return limpio, registro

In [ ]:
limpio, registro = limpiar_datos(sucio)
registro

In [ ]:
# ¿El resultado coincide con la base limpia original?
iguales = (limpio.sort_values("id")["nota_final"].values ==
           alumnos.sort_values("id")["nota_final"].values).all()
print("nota_final coincide con alumnos.csv:", iguales)
limpio.head()

---
## ✍️ Ejercicios — Parte 2 (`ejercicios_parte2.py`)
Completa cada función reemplazando `pass` y ejecuta la celda de **comprobación**: cada ejercicio correcto muestra ✅.

In [ ]:
def ejercicio_1_agregar_profesor(alumnos, cursos):
    """Une alumnos con cursos (profesor, sala, creditos). 20 filas, relación muchos-a-uno."""
    pass  # TODO

def ejercicio_2_asistencia_promedio(alumnos, asistencia):
    """Promedia la asistencia mensual por alumno y únela en la columna 'asistencia_prom'. 20 filas."""
    pass  # TODO

def ejercicio_3_historico(alumnos, periodo2):
    """Apila ambos períodos con una columna 'periodo' ('2026-1' / '2026-2'), índice 0..n-1."""
    pass  # TODO

def ejercicio_4_alumnos_que_no_continuan(alumnos, periodo2):
    """Lista con los NOMBRES de alumnos del período 1 que no están en el período 2 (por 'id')."""
    pass  # TODO

def ejercicio_5_limpiar(sucio):
    """Limpia alumnos_sucio: 20 filas, 3 cursos, nota_final y asistencia numéricas."""
    pass  # TODO

In [ ]:
# Comprobación automática (no modificar)
def test_1():
    r = ejercicio_1_agregar_profesor(alumnos, cursos)
    assert len(r) == 20 and "profesor" in r.columns and r["profesor"].notna().all()
def test_2():
    r = ejercicio_2_asistencia_promedio(alumnos, asistencia)
    assert len(r) == 20 and abs(r.loc[r["id"] == 1, "asistencia_prom"].iloc[0] - 95) < 0.01
def test_3():
    r = ejercicio_3_historico(alumnos, periodo2)
    assert len(r) == 40 and set(r["periodo"]) == {"2026-1", "2026-2"} and list(r.index) == list(range(40))
def test_4():
    assert sorted(ejercicio_4_alumnos_que_no_continuan(alumnos, periodo2)) == ["Diego Fernandez", "Hugo Vega"]
def test_5():
    r = ejercicio_5_limpiar(sucio)
    assert len(r) == 20 and sorted(r["curso"].unique()) == ["Fisica", "Matematicas", "Quimica"]
    assert pd.api.types.is_numeric_dtype(r["nota_final"]) and pd.api.types.is_numeric_dtype(r["asistencia"])

def comprobar():
    for n, t in enumerate([test_1, test_2, test_3, test_4, test_5], start=1):
        try:
            t(); print(f"Ejercicio {n}: ✅ OK")
        except Exception as e:
            print(f"Ejercicio {n}: ⏳ pendiente o incorrecto ({type(e).__name__})")

comprobar()

---
## ✅ Soluciones (revisión en clase)
> 💡 Colapsa este título para ocultar las soluciones hasta la revisión.

In [ ]:
def ejercicio_1_agregar_profesor(alumnos, cursos):
    # how="left" conserva los 20 alumnos; validate evita duplicados silenciosos
    return alumnos.merge(cursos, on="curso", how="left", validate="many_to_one")

def ejercicio_2_asistencia_promedio(alumnos, asistencia):
    # 1) bajar la granularidad (id, mes) -> (id)   2) unir 1 a 1
    prom = (asistencia.groupby("id", as_index=False)["asistencia"].mean()
            .rename(columns={"asistencia": "asistencia_prom"}))
    return alumnos.merge(prom, on="id", how="left", validate="one_to_one")

def ejercicio_3_historico(alumnos, periodo2):
    return pd.concat([alumnos.assign(periodo="2026-1"),
                      periodo2.assign(periodo="2026-2")], ignore_index=True)

def ejercicio_4_alumnos_que_no_continuan(alumnos, periodo2):
    cruce = alumnos.merge(periodo2[["id"]], on="id", how="left", indicator=True)
    return cruce.loc[cruce["_merge"] == "left_only", "nombre"].tolist()
    # Alternativa: alumnos.loc[~alumnos["id"].isin(periodo2["id"]), "nombre"].tolist()

def ejercicio_5_limpiar(sucio):
    # Reutiliza el flujo de la sección 5.3
    limpio, _ = limpiar_datos(sucio)
    return limpio

comprobar()

**Resultados esperados:** (1) 20 filas con profesor · (2) Ana Torres: asistencia promedio 95 ·
(3) 40 filas, 20 por período · (4) Diego Fernandez y Hugo Vega · (5) 20 filas, 3 cursos, columnas numéricas.

---
### ➡️ Próximo módulo: uso estadístico con Python — regresión lineal simple y múltiple.